# Titanic Dataset Features:
# survived: Whether the passenger survived (0 = No, 1 = Yes).
# pclass: Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd).
# sex: Gender of the passenger.
# age: Age of the passenger in years. Some values are missing.
# sibsp: Number of siblings/spouses aboard the Titanic.
# parch: Number of parents/children aboard the Titanic.
# fare: Passenger fare.
# embarked: Port of embarkation (C = Cherbourg; Q = Queenstown; S = Southampton).
# class: Duplicate of 'pclass' (used for plotting by Seaborn).
# who: Describes whether the passenger is a man, woman, or child.
# adult_male: Indicates whether the passenger is an adult male (True/False).
# deck: The deck the passenger was on (missing for many passengers).
# embark_town: The name of the town where the passenger boarded.
# alive: Indicator of whether the passenger survived (Yes/No, derived from 'survived').
# alone: Indicates whether the passenger was traveling alone (True/False).

In [2]:
import pandas as pd
import sklearn
from sklearn import model_selection
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math

In [4]:
# Load the dataset

titanicdata = "titanic.csv"
titanicdf = pd.read_csv(titanicdata)
titanicdf.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
titanicdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [6]:
titanicdf.select_dtypes(include = 'object').columns
# titanicdf.select_dtypes(include = 'object').columns.tolist()
# titanicdf.select_dtypes(include = ['object', 'int64'])
# titanicdf.select_dtypes(include = ['object', 'int64'])[:5]
# titanicdf.select_dtypes(exclude = ['object'])[:5]

C:\Users\RU20688694\AppData\Local\Temp\ipykernel_19772\3610466420.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  titanicdf.select_dtypes(include = 'object').columns


Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='str')

In [7]:
titanicdf['Embarked'].value_counts()
# titanicdf['Embarked'].isna().sum()
# titanicdf['Age'].isna().sum()
# titanicdf['Cabin'].isna().sum()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [8]:
# Pre Processing
titanicdf.Embarked = titanicdf.Embarked.fillna(titanicdf['Embarked'].mode()[0])
# mean : average of the data
# median : middle value of the data
# mode : most frequently occurring value in the data

In [9]:
median_age = titanicdf.Age.median()
median_fare = titanicdf.Fare.median()
print(median_age)
print(median_fare)

titanicdf.Age.fillna(median_age, inplace = True)
titanicdf.Fare.fillna(median_fare, inplace = True)

28.0
14.4542


C:\Users\RU20688694\AppData\Local\Temp\ipykernel_19772\4249398157.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  titanicdf.Age.fillna(median_age, inplace = True)
C:\Users\RU20688694\AppData\Local\Temp\ipykernel_19772\4249398157.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained 

0       7.2500
1      71.2833
2       7.9250
3      53.1000
4       8.0500
        ...   
886    13.0000
887    30.0000
888    23.4500
889    30.0000
890     7.7500
Name: Fare, Length: 891, dtype: float64

In [10]:
titanicdf.drop('Cabin', axis = 1,inplace = True)

In [11]:
titanicdf['FamilySize'] = titanicdf['SibSp'] + titanicdf['Parch']+1
titanicdf['GenderClass'] = titanicdf.apply(lambda x: 'child' if x['Age'] < 15 else x['Sex'],axis=1)

In [12]:
# titanicdf[titanicdf.Age<15].head(2)
titanicdf['GenderClass'].value_counts()
# titanicdf['Embarked'].value_counts()

GenderClass
male      538
female    275
child      78
Name: count, dtype: int64

In [13]:
# titanicdf["Pclass"].unique()
titanicdf.Pclass.unique()
# set(titanicdf.Pclass.unique())

array([3, 1, 2])

In [14]:
titanicdf.groupby("Pclass").count()

,PassengerId,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,FamilySize,GenderClass
Pclass,,,,,,,,,,,,
1,216,216,216,216,186,216,216,216,216,216,216,216
2,184,184,184,184,173,184,184,184,184,184,184,184
3,491,491,491,491,355,491,491,491,491,491,491,491


In [15]:
# dict(titanicdf.groupby('Pclass')['PassengerId'].count())

In [16]:
 # titanicdf.groupby('Embarked').count()

In [17]:
titanicdf.groupby('Survived').count()
# titanicdf.groupby('Survived')['PassengerId'].count()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,FamilySize,GenderClass
Survived,,,,,,,,,,,,
0,549,549,549,549,424,549,549,549,549,549,549,549
1,342,342,342,342,290,342,342,342,342,342,342,342


In [18]:
# frequency of unique values
titanicdf['Survived'].value_counts()
# titanicdf['Embarked'].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

In [19]:
# Check for missing values
print("Missing values in each column:")
print(titanicdf.isnull().sum())

Missing values in each column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Embarked         0
FamilySize       0
GenderClass      0
dtype: int64


In [20]:
titanicdf.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,FamilySize,GenderClass
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,2,male
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,2,female
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,1,female


In [21]:
titanicdf = pd.get_dummies(titanicdf, columns=['GenderClass','Embarked'], drop_first=True)
titanicdf.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,FamilySize,GenderClass_female,GenderClass_male,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,2,False,True,False,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,2,True,False,False,False
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,1,True,False,False,True


In [22]:
# titanicdf1 = titanicdf.drop(['Name','Ticket','Sex','SibSp','Parch', 'Fare_Range'], axis = 1)
# titanicdf1.head()
titanicdf1 = titanicdf.drop(['Name', 'Sex', 'Ticket'], axis = 1)
X = titanicdf1.drop("Survived", axis=1)
y = titanicdf1["Survived"]
# X.head()
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [24]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
X_train.isnull().sum()

PassengerId             0
Pclass                  0
Age                   140
SibSp                   0
Parch                   0
Fare                    0
FamilySize              0
GenderClass_female      0
GenderClass_male        0
Embarked_Q              0
Embarked_S              0
dtype: int64

In [28]:
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
X_test['Age'] = X_test['Age'].fillna(X_train['Age'].median())
 

In [29]:
X_train.isnull().sum()

PassengerId           0
Pclass                0
Age                   0
SibSp                 0
Parch                 0
Fare                  0
FamilySize            0
GenderClass_female    0
GenderClass_male      0
Embarked_Q            0
Embarked_S            0
dtype: int64

In [30]:
model = LogisticRegression(solver = 'liblinear')
model.fit(X_train, y_train)

,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some pen

In [31]:
predictions = model.predict (X_test)
# Uses a threshold (default = 0.5): >= 0.5 → Survived (1), < 0.5 → Not survived (0)
# Confusion Metrics
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, model.predict(X_test))

array([[90, 15],
       [23, 51]])

In [32]:
report = classification_report (y_test, predictions)
print (report)

              precision    recall  f1-score   support

           0       0.80      0.86      0.83       105
           1       0.77      0.69      0.73        74

    accuracy                           0.79       179
   macro avg       0.78      0.77      0.78       179
weighted avg       0.79      0.79      0.79       179



In [33]:
model.coef_
# Positive coefficient → increases survival probability
# Negative coefficient → decreases survival probability
# Example interpretation:
# Sex_female = +2.1 → Being female strongly increases survival chance
# Pclass_3 = -1.3 → 3rd class reduces survival probability

array([[ 4.55124137e-04, -7.29337494e-01, -1.02805773e-02,
        -9.42882888e-01, -7.77707357e-01,  5.01592420e-03,
         5.60619629e-01,  5.75683861e-01, -2.49686470e+00,
        -2.39507002e-01, -3.23974120e-01]])

In [34]:
X_test.head()

,PassengerId,Pclass,Age,SibSp,Parch,Fare,FamilySize,GenderClass_female,GenderClass_male,Embarked_Q,Embarked_S
709,710,3,28.0,1,1,15.2458,3,False,True,False,False
439,440,2,31.0,0,0,10.5000,1,False,True,False,True
840,841,3,20.0,0,0,7.9250,1,False,True,False,True
720,721,2,6.0,0,1,33.0000,2,False,False,False,True
39,40,3,14.0,1,0,11.2417,2,False,False,False,False


In [35]:
y_test.head()

709    1
439    0
840    0
720    1
39     1
Name: Survived, dtype: int64